In [5]:
#| default_exp envs.POMDPs

In [6]:
#| hide
%load_ext autoreload
%autoreload 2

from nbdev import show_doc
from projective_simulation.envs.core import Abstract_Env
import numpy as np

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# POMDPs

## General Constructor

In [19]:
#| export
class POMDP(Abstract_Env):
    """
    A general constructor for POMDPs
    """
    def __init__(self,
                 percepts: np.ndarray,             #An SxK array where S is the number of Percepts and K is the number of categories for each percept. If 1d, converted to Sx1
                 observation_function: np.ndarray, #An NxS array, where N is the number of states in the cycle and S is the number of possible percepts. Rows contain probability distributions
                 transition_function: np.ndarray,  #An NxNxA array, where A is the number of actions. Rows in each slice contain probability distributions.
                 initial_state: int = 0            #Start state of POMDP
                ):
        '''
        Checks validity of observation and transition function, assigns variables to self.
        '''
        assert isinstance(percepts, np.ndarray)
        if percepts.ndim == 1:
            percepts = percepts[:,np.newaxis]
        assert np.shape(percepts)[0] == np.shape(observation_function)[1]
        assert np.shape(transition_function)[0] == np.shape(observation_function)[0]
        assert np.shape(transition_function)[0] == np.shape(transition_function)[1]
        assert transition_function.ndim == 3
        assert np.isclose(np.sum(transition_function,axis =1), 1, atol=1e-9).all() #all rows sum to 1
        assert np.isclose(np.sum(observation_function,axis =1), 1, atol=1e-9).all() #all rows sum to 1
        assert initial_state in range(np.shape(observation_function)[0])
         
        self.percepts = percepts
        self.observation_function = observation_function
        self.transition_function = transition_function
        state = initial_state
        super().__init__(state = state)

    def transition(self, action):
        """
        randomly select a new state using transition probabilites from current state and action
        """
        if not action in range(np.shape(self.transition_function)[2]):
            raise ValueError("The action input for POMDP transition must be integer valued and within the scope of the transition function")
        transition_probs = self.transition_function[self.state,:, action]
        self.state = np.random.choice(len(transition_probs), p = transition_probs)

    def get_observation(self):
        """
        randomly select a percept using observation probabilites from current state
        """
        percept_probs = self.observation_function[self.state,:]
        percept_index = np.random.choice(len(percept_probs), p = percept_probs)
        return self.percepts[percept_index,:]

In [21]:
show_doc(POMDP, name = "POMDP")

---

### POMDP

>      POMDP (percepts:numpy.ndarray, observation_function:numpy.ndarray,
>             transition_function:numpy.ndarray, initial_state:int=0)

*A general constructor for POMDPs*

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| percepts | ndarray |  | An SxK array where S is the number of Percepts and K is the number of categories for each percept. If 1d, converted to Sx1 |
| observation_function | ndarray |  | An NxS array, where N is the number of states in the cycle and S is the number of possible percepts. Rows contain probability distributions |
| transition_function | ndarray |  | An NxNxA array, where A is the number of actions. Rows in each slice contain probability distributions. |
| initial_state | int | 0 | Start state of POMDP |

### POMDP Example

Here we define a simple Markov Decision Process in which an agent must match one of two actions to the first category of its percept. If it does, the environment moves to a state that produces an observable reward. If it does, the environment moves to a state without reward. Note that this environment is fully observable - each state maps to a unique percept. The POMDP class is general and can handle such a case.

In [10]:
from projective_simulation.agents.core import Basic_2Layer_Agent

#Create a percept for each combination of signal and reward
percepts = np.array([[0,0],
                     [0,1],
                     [1,0],
                     [1,1]]
                   )
#Create observation function for 4-state environment, each selects producing one of the perceprs
observation_function = np.array([[1.,0.,0.,0.],
                                 [0.,1.,0.,0.],
                                 [0.,0.,1.,0.],
                                 [0.,0.,0.,1.]]                                
                               )
#Create transition function for 4-state, 2-action environment, where transitions are uniform-random to states with the appropriate reward
transition_function = np.array([[[0.,0.5], #in state 0, action 0 transition either to state 1 or 3. Action 1 transitions either to state 0 or 2.
                                 [0.5,0.],
                                 [0.,0.5],
                                 [0.5,0.]],
                                [[0.,0.5], #in state 1, action 0 transition either to state 1 or 3. Action 1 transitions either to state 0 or 2.
                                 [0.5,0.],
                                 [0.,0.5],
                                 [0.5,0.]],
                                [[0.5,0.], #in state 2, action 0 transition either to state 0 or 2. Action 1 transitions either to state 1 or 3.
                                 [0.,0.5],
                                 [0.5,0.],
                                 [0.,0.5]],
                                [[0.5,0.], #in state 3, action 0 transition either to state 0 or 2. Action 1 transitions either to state 1 or 3.
                                 [0.,0.5],
                                 [0.5,0.],
                                 [0.,0.5]]]
                              )
#Initialize POMDP
example_env = POMDP(percepts = percepts, observation_function = observation_function, transition_function = transition_function)

#Initialize Agent
test_agent = Basic_2Layer_Agent(num_actions = np.shape(transition_function)[2], glow = 1., policy = 'softmax', policy_parameters = 1)

#run percept action loop for 50 steps
T = 100
percepts = [None] * T #empty list for storing percepts
actions = [None] * T  #empty list for storing actions
for t in range(T):
    percept = example_env.get_observation()
    percepts[t] = percept
    reward = percept[1]
    test_agent.update(reward)
    action = test_agent.deliberate(str(percept))
    actions[t] = action
    example_env.transition(action)


#show final 10 percepts and actions. Agent should have learned to select actions that produce rewards
for t in range(T-10,T):
    print(f'Step {t}: percept = {percepts[t]}; action = {actions[t]}')

Step 90: percept = [1 1]; action = 1
Step 91: percept = [0 1]; action = 0
Step 92: percept = [0 1]; action = 0
Step 93: percept = [1 1]; action = 1
Step 94: percept = [1 1]; action = 1
Step 95: percept = [1 1]; action = 1
Step 96: percept = [1 1]; action = 1
Step 97: percept = [0 1]; action = 0
Step 98: percept = [0 1]; action = 0
Step 99: percept = [0 1]; action = 0


### Prebuilt POMDPs



In [17]:
#|export
class Cyclic_Env(POMDP):
    """
    An environment that cycles deterministically through a sequence of percepts that may be passed to an agent
    """
    def __init__(self,
                 percept_cycle: np.ndarray,     #an N x K array, where N is the number of states in the cycles and K is the number of categories in a percept. If 1d, will be converted to Nx1
                 initial_state: int = 0,        #index of first percept returned by get_observation()
                 supress_warning: bool = False  #warns if Action != 0 is input to transition function
                ):
        '''
        Constructs deterministic transition function and observation from percept_cycle
        '''
        if percept_cycle.ndim == 1:
            percept_cycle = percept_cycle[:, np.newaxis]
        
        # Get unique rows and, for each row of percept_cycle, the index of its unique representative
        percepts, inverse = np.unique(percept_cycle, axis=0, return_inverse=True)
        
        # One-hot encode those indices to form the observation function
        observation_function = np.eye(percepts.shape[0], dtype=int)[inverse]       

        #create a cyclic shifted diagonal matrix
        S = np.shape(percept_cycle)[0]
        transition_function = np.roll(np.eye(N = S), shift = 1, axis = 1) 
        transition_function = transition_function[:,:,np.newaxis] #adds a dimension to allow for a single action
        
        super().__init__(percepts = percepts, observation_function = observation_function, transition_function = transition_function, initial_state = initial_state)
        self.supress_warning = supress_warning
        
    def transition(self, action):
        if not action == 0 and not self.supress_warning:
            print('Warning: Cyclic_Env is not action mediated. Action input has been converted to 0')
        action = 0
        super().transition(action)

In [18]:
show_doc(Cyclic_Env, name = "Cyclic_Env")

---

### Cyclic_Env

>      Cyclic_Env (percept_cycle:numpy.ndarray, initial_state:int=0,
>                  supress_warning:bool=False)

*An environment that cycles deterministically through a sequence of percepts that may be passed to an agent*

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| percept_cycle | ndarray |  | an N x K array, where N is the number of states in the cycles and K is the number of categories in a percept. If 1d, will be converted to Nx1 |
| initial_state | int | 0 | index of first percept returned by get_observation() |
| supress_warning | bool | False | warns if Action != 0 is input to transition function |